In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [2]:
# ── Activation functions ──────────────────────────
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(z):
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


In [3]:
# ── Loss function ─────────────────────────────────
def cross_entropy_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))




In [5]:
# ── Neural Network class ──────────────────────────
class NeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.01):
        self.lr  = learning_rate
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes) - 1):
            # He initialization
            W = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(W)
            self.biases.append(b)

    def forward(self, X):
        self.activations = [X]
        self.z_values    = []
        A = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            Z = A @ W + b
            self.z_values.append(Z)
            # ReLU for hidden layers, softmax for output
            A = relu(Z) if i < len(self.weights) - 1 else softmax(Z)
            self.activations.append(A)
        return A

    def backward(self, X, y_true):
        m     = X.shape[0]
        grads_w = [None] * len(self.weights)
        grads_b = [None] * len(self.biases)

        # Output layer gradient (softmax + cross-entropy combined)
        delta = self.activations[-1] - y_true

        for i in reversed(range(len(self.weights))):
            grads_w[i] = self.activations[i].T @ delta / m
            grads_b[i] = np.mean(delta, axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.weights[i].T) * relu_deriv(self.z_values[i-1])



In [10]:
# ── Neural Network class ──────────────────────────
class NeuralNetwork:
    def __init__(self, layer_sizes, learning_rate=0.01):
        self.lr  = learning_rate
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes) - 1):
            # He initialization
            W = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(W)
            self.biases.append(b)

    def forward(self, X):
        self.activations = [X]
        self.z_values    = []
        A = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            Z = A @ W + b
            self.z_values.append(Z)
            # ReLU for hidden layers, softmax for output
            A = relu(Z) if i < len(self.weights) - 1 else softmax(Z)
            self.activations.append(A)
        return A

    def backward(self, X, y_true):
        m     = X.shape[0]
        grads_w = [None] * len(self.weights)
        grads_b = [None] * len(self.biases)

        # Output layer gradient (softmax + cross-entropy combined)
        delta = self.activations[-1] - y_true

        for i in reversed(range(len(self.weights))):
            grads_w[i] = self.activations[i].T @ delta / m
            grads_b[i] = np.mean(delta, axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.weights[i].T) * relu_deriv(self.z_values[i-1])

        # Update weights
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * grads_w[i]
            self.biases[i]  -= self.lr * grads_b[i]

    def train(self, X, y, epochs=1000):
        losses = []
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss   = cross_entropy_loss(y, y_pred)
            losses.append(loss)
            self.backward(X, y)
            if epoch % 100 == 0:
                acc = self.accuracy(X, y)
                print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Acc: {acc:.4f}")
        return losses

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def accuracy(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == np.argmax(y, axis=1))


In [11]:
# ── Prepare data ──────────────────────────────────
iris    = load_iris()
X, y    = iris.data, iris.target.reshape(-1, 1)

enc     = OneHotEncoder(sparse_output=False)
y_onehot = enc.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [12]:
# ── Build & train ─────────────────────────────────
# Architecture: 4 inputs → 8 hidden → 3 outputs
nn = NeuralNetwork(layer_sizes=[4, 8, 3], learning_rate=0.05)
losses = nn.train(X_train, y_train, epochs=1000)


Epoch    0 | Loss: 0.9865 | Acc: 0.6750
Epoch  100 | Loss: 0.2899 | Acc: 0.9000
Epoch  200 | Loss: 0.2202 | Acc: 0.9250
Epoch  300 | Loss: 0.1704 | Acc: 0.9417
Epoch  400 | Loss: 0.1368 | Acc: 0.9500
Epoch  500 | Loss: 0.1157 | Acc: 0.9500
Epoch  600 | Loss: 0.1020 | Acc: 0.9500
Epoch  700 | Loss: 0.0925 | Acc: 0.9500
Epoch  800 | Loss: 0.0856 | Acc: 0.9500
Epoch  900 | Loss: 0.0802 | Acc: 0.9500
